# Point Cloud to CAD-sequence

In this notebook the complete interactive pipeline for encoding point clouds into a latent space, from which DeepCAD decodes a CAD-sequence.

In [1]:
import os
import sys
import shutil
import glob
import json
import argparse
import importlib

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn.functional as F

from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import read_step_file, write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import vec2CADsolid, create_CAD
from models.DeepCAD.utils.file_utils import ensure_dir

### PointNet++

In [2]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

def load_pointnet():
    sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
    model = importlib.import_module('pointnet2_cls_ssg')
    classifier = model.get_model(latent_dim, normal_channel=False)
    criterion = model.get_loss_mse()
    classifier.apply(inplace_relu) 
    
    saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    state_dict = saved_model['model_state_dict']
    if 'module.' in next(iter(state_dict)):
        monitor.log_and_print("Model was saved wrapped in nn.DataParallel.\nRemoving 'module.' from state dict.")
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}#
    classifier.eval()
    classifier.load_state_dict(state_dict)
    print(f"Loading PointNet++ from {os.path.abspath(model_path)}")
    return classifier

### DeepCAD

In [3]:
def load_deepcad(cfg):
    tr_agent = TrainerAE(cfg)
    tr_agent.load_ckpt(cfg.ckpt)
    tr_agent.net.eval()
    return tr_agent

### Load data

In [4]:
def get_data(indices, dataset):
    pc_list = []
    lat_rep_list = []  
    pc_paths = []
    cad_seq_list = []
    pc_dir = os.path.join(results_dir, "infered_point_clouds")
    if os.path.exists(pc_dir):
        shutil.rmtree(pc_dir)
    os.mkdir(pc_dir)
    
    for i in indices:
        pc, lat_rep, cad_seq = dataset[i]
        pc_path = os.path.abspath(dataset.get_pc_path(i))
        pc_path_destination = os.path.join(pc_dir, os.path.basename(pc_path))
        shutil.copy2(pc_path, pc_path_destination)
        pc_paths.append(pc_path_destination)
        pc_list.append(pc)
        lat_rep_list.append(lat_rep)
        cad_seq_list.append(cad_seq)

    with h5py.File(h5_file, 'a') as hf:
        dt = h5py.special_dtype(vlen=str)
        path_dataset = hf.create_dataset("pc_paths", shape=(len(pc_paths),), dtype=dt)
        path_dataset[:] = pc_paths
        
    pc_batch = torch.stack(pc_list, dim=0)
    lat_rep_batch = torch.stack(lat_rep_list, dim=0)
    cad_seq_batch = torch.stack(cad_seq_list, dim=0)
    
    return pc_batch, lat_rep_batch, cad_seq_batch

### Inference

In [5]:
def infer_pointnet(indices, dataset, model):
    with h5py.File(h5_file, 'w') as hf:
        z_pred = hf.create_dataset('z_pred', 
                                   shape=(len(indices), latent_dim), 
                                   dtype=np.float32)
        z_target = hf.create_dataset('z_target',
                                     shape=(len(indices), latent_dim),
                                     dtype = np.float32)
        seq_target = hf.create_dataset('seq_target', 
                                       shape=(len(indices), cfg.max_total_len, cfg.n_args + 1), 
                                       dtype=np.int64)
        
        pc, lat_rep, cad_seq = get_data(indices, dataset)
        z_target[:] = lat_rep
        seq_target[:] = cad_seq

        criterion_loader = importlib.import_module('pointnet2_cls_ssg')
        criterion = criterion_loader.get_loss_mse()
        
        with torch.no_grad():
            pc = pc.transpose(2, 1)
            pred, _ = model(pc)
            z_pred[:] = pred.detach()
            loss = criterion(pred, lat_rep)
            print(f"Avg. MSE-Loss: {loss.detach().item():.8e}")
            return pred, cad_seq

In [6]:
def infer_deepcad(pred, cad_seq, tr_agent):
    with h5py.File(h5_file, 'a') as hf:
        seq_pred = hf.create_dataset('seq_pred', 
                                     shape=(pred.shape[0], cfg.max_total_len, cfg.n_args + 1), 
                                     dtype=np.int64)
        cmd_logits = hf.create_dataset('cmd_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_commands), 
                                       dtype=np.float32)
        args_logits = hf.create_dataset('args_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_args, cfg.args_dim + 1), 
                                       dtype=np.float32)
        with torch.no_grad():
            pred = pred.unsqueeze(dim = 1)
            output = tr_agent.decode(pred)

            output["tgt_commands"] = cad_seq[:, :, 0] 
            output["tgt_args"] = cad_seq[:, :, 1:]
            loss_dict = tr_agent.loss_func(output)
            
            batch_out_vec = tr_agent.logits2vec(output)
            
            cmd_logits[:] = output['command_logits']
            args_logits[:] = output['args_logits']
            seq_pred[:] = batch_out_vec
            
            print(f"Avg. Command-Loss: {loss_dict['loss_cmd'].detach().cpu().item():.8e}")
            print(f"Avg. Argument-Loss: {loss_dict['loss_args'].detach().cpu().item():.8e}")

### Utils

In [7]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=-1, keepdims=True)

In [117]:
def cross_entropy(logits, target):
    logits = torch.from_numpy(logits).unsqueeze(0)
    target = torch.tensor([target]).long()
    print(logits.shape, target.shape)
    return F.cross_entropy(logits, target)

### Visualization

In [137]:
def show_results(idx):
    with h5py.File(h5_file, "r") as hf:
        pc_path = hf['pc_paths'][idx].decode("utf-8")
        args_logits = hf['args_logits'][idx]
        cmd_logits = hf['cmd_logits'][idx]
        seq_pred = hf['seq_pred'][idx]
        seq_target = hf['seq_target'][idx]
        z_pred = hf['z_pred'][idx]
        z_target = hf['z_target'][idx]

    ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
    LINE_IDX = ALL_COMMANDS.index('Line')
    ARC_IDX = ALL_COMMANDS.index('Arc')
    CIRCLE_IDX = ALL_COMMANDS.index('Circle')
    EOS_IDX = ALL_COMMANDS.index('EOS')
    SOL_IDX = ALL_COMMANDS.index('SOL')
    EXT_IDX = ALL_COMMANDS.index('Ext')

    print(f"Point Cloud path: {pc_path}")
   # print(args_logits.shape, cmd_logits.shape, seq_pred.shape, seq_target.shape, z_pred.shape, z_target.shape)
    for idx, command in enumerate(ALL_COMMANDS):
        print(f"{idx} -> {command}")

    target_commands = []
    predicted_commands = []
    pred_commands_prob = []
    cmd_loss = []
    cmd_loss_torch = []
    target_commands_prob = []
    
    all_pred_commands = list(seq_pred[:, 0])
    seq_length = list(seq_target[:, 0]).index(EOS_IDX) + 3
    # print(cmd_logits[:seq_length, :])
    cmd_logits_softmax = softmax(cmd_logits[:seq_length, :])
    
    for i in range(seq_length):
        predicted_commands.append(int(all_pred_commands[i]))
        target_commands.append(int(seq_target[i, 0]))
        pred_commands_prob.append(round(float(cmd_logits_softmax[i, predicted_commands[i]]) * 100, 5))
        cmd_loss.append(cross_entropy(cmd_logits[i,:], target_commands[i]).item())
        target_commands_prob.append(round(float(cmd_logits_softmax[i, target_commands[i]]) * 100, 5))
    
    df = pd.DataFrame(list(zip(target_commands, predicted_commands, target_commands_prob, pred_commands_prob, cmd_loss)),
                      columns=['trgt', 'pred','prob_trgt', 'prob_pred', 'loss'])
    print(f"Sum CADLoss for {seq_length} commands:  {sum(cmd_loss):.8e}")
    print(f"Mean CADLoss for {seq_length} commands: {np.mean(cmd_loss):.8e}")
    return df

### Export to .step, .stl and .obj

In [10]:
def export2step():
    form = "h5"
    filter = True
    output_dir = os.path.join(results_dir, "step_files")
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.mkdir(output_dir)
    h5_path = os.path.join(results_dir, "data.h5")

    with h5py.File(h5_path, 'r') as fp:
        out_vec = fp['seq_pred'][:].astype(np.float64)
        names = fp['pc_paths'][:]
        print(out_vec.shape)
        for i, seq in enumerate(out_vec):
            pc_path = names[i].decode('utf-8')
            out_shape = vec2CADsolid(seq)
    
            if filter:
                analyzer = BRepCheck_Analyzer(out_shape)
                if not analyzer.IsValid():
                    print(f"CAD-sequence of {os.path.basename(pc_path)} is invalid.")
                    continue
    
            pc_name = os.path.splitext(os.path.basename(pc_path))[0]
            save_path = os.path.join(output_dir, pc_name + ".step")
            write_step_file(out_shape, save_path)


In [11]:
def step2stl():
    step_files = glob.glob(os.path.join(results_dir, "step_files", "*.step"))
    obj_dir = os.path.join(results_dir, "stl_files")
    if os.path.exists(obj_dir):
        shutil.rmtree(obj_dir)
    os.mkdir(obj_dir)

    for step_file in step_files:

        save_path = os.path.join(obj_dir, os.path.splitext(os.path.basename(step_file))[0] + ".stl")
    
        step_reader = STEPControl_Reader()
        step_reader.ReadFile(step_file)
        step_reader.TransferRoots()
        shape = step_reader.OneShape()

        BRepMesh_IncrementalMesh(shape, 0.5)

        stl_writer = StlAPI_Writer()
        stl_writer.Write(shape, save_path)

In [12]:
def step2obj(): # .obj files from this function lead to malformed file error in cloud compare
    step_files = glob.glob(os.path.join(results_dir, "step_files", "*.step"))
    obj_dir = os.path.join(results_dir, "obj_files")
    if os.path.exists(obj_dir):
        shutil.rmtree(obj_dir)
    os.mkdir(obj_dir)
    
    for step_file in step_files:
        shape = read_step_file(step_file)
        save_path = os.path.join(obj_dir, os.path.splitext(os.path.basename(step_file))[0] + ".obj")
        write_step_file(shape, save_path)

## Start

### Variables

Store the models in ```experiments```, a results directory will be created for each respective model.

In [53]:
model_name = "best_5"

In [54]:
model_path = os.path.join("experiments", model_name) + ".pth"
results_dir = os.path.join("experiments", model_name + "_results")
if not os.path.exists(results_dir):
    os.mkdir(results_dir)
h5_file = os.path.join(results_dir, "data.h5")
cfg = ConfigAE('test', model_path="../data/latent")
latent_dim = 256

In [55]:
pointnet_plusplus = load_pointnet()
deepcad = load_deepcad(cfg)

Loading PointNet++ from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/notebooks/experiments/best_5.pth
Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model/ckpt_epoch1000.pth ...


In [56]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')
print(f"Dataset contains {len(dataset)} samples.")

Dataset contains 8038 samples.


In [76]:
indices = [0]

In [77]:
pred, trgt_cad_seq = infer_pointnet(indices, dataset, pointnet_plusplus)

Avg. MSE-Loss: 1.01912776e-02


In [78]:
infer_deepcad(pred, trgt_cad_seq, deepcad)

Avg. Command-Loss: 9.53670394e-07
Avg. Argument-Loss: 8.39450877e-06


In [138]:
show_results(0)

Point Cloud path: experiments/best_5_results/infered_point_clouds/00250456.ply
0 -> Line
1 -> Arc
2 -> Circle
3 -> EOS
4 -> SOL
5 -> Ext
[[  2.1661167   -0.5399175   -2.4103343  -18.082228    14.250947
    1.2798617 ]
 [ 16.941368   -13.5415535   -9.801673   -10.592526    -5.635955
   -9.183432  ]
 [ 17.89241     -9.9130125  -16.788755   -10.442123    -5.5110526
  -10.155578  ]
 [ 19.133535    -7.507553   -16.81719     -5.6530304  -12.67111
  -11.46177   ]
 [ 15.5345955   -8.302415   -10.729546    -6.5964565  -10.851784
   -8.540977  ]
 [ -0.94262093 -10.12298    -12.208171     1.4186146    1.1451744
   18.222597  ]
 [ -4.9011273  -26.194721    -9.722097    47.24709     12.212529
    3.5664697 ]
 [ -1.7283378  -23.2751      -9.528018    47.1975      -5.6106853
    1.3542323 ]
 [ -0.5251464  -24.51542    -12.643808    46.52833     -2.2675776
    4.1713204 ]]
torch.Size([1, 6]) torch.Size([1])
torch.Size([1, 6]) torch.Size([1])
torch.Size([1, 6]) torch.Size([1])
torch.Size([1, 6]) torch.

,trgt,pred,prob_trgt,prob_pred,loss
0,4,4,99.99916,99.99916,8.463824e-06
1,0,0,100.00000,100.00000,0.000000e+00
2,0,0,100.00000,100.00000,0.000000e+00
3,0,0,100.00000,100.00000,0.000000e+00
4,0,0,100.00000,100.00000,0.000000e+00
5,5,5,99.99999,99.99999,1.192093e-07
6,3,3,100.00000,100.00000,0.000000e+00
7,3,3,100.00000,100.00000,0.000000e+00
8,3,3,100.00000,100.00000,0.000000e+00


Observation:

- If arg 223 is the target, in the logits it has the index 224, because we shift every index by +1 because of the -1 pad value
- Otherwise the -1 pad value would lead to choosing the last element

In [109]:
lol = torch.tensor([[2.3814e-28, 1.9496e-28, 2.0373e-28, 2.0855e-28, 1.2073e-28, 2.3458e-28,
         1.8129e-28, 1.5884e-28, 2.2812e-28, 1.5654e-28, 1.6599e-28, 1.9552e-28,
         1.6958e-28, 2.4901e-28, 2.1314e-28, 1.5340e-28, 3.3773e-28, 1.6066e-28,
         3.0462e-28, 2.4224e-28, 1.7162e-28, 1.7156e-28, 3.7628e-28, 2.1147e-28,
         1.4206e-28, 1.9673e-28, 1.3825e-28, 1.9837e-28, 2.1591e-28, 1.9434e-28,
         1.3217e-28, 1.6975e-28, 1.7440e-28, 2.0688e-28, 1.6297e-28, 1.5500e-28,
         1.4088e-28, 9.8359e-34, 1.7934e-31, 2.2875e-28, 1.4415e-28, 1.9848e-28,
         1.5285e-28, 1.5219e-28, 2.2277e-28, 3.7884e-35, 2.2574e-37, 2.8830e-39,
         8.9917e-30, 1.9856e-28, 1.5100e-37, 2.4738e-34, 1.6631e-31, 6.4733e-39,
         1.7967e-28, 1.9681e-31, 2.8166e-43, 2.4141e-37, 2.7439e-28, 2.8435e-28,
         1.8491e-28, 5.7249e-34, 5.1628e-32, 4.9201e-37, 2.2386e-28, 4.2326e-39,
         0.0000e+00, 8.1556e-42, 2.4549e-39, 1.3698e-31, 1.6571e-28, 3.0242e-35,
         1.0546e-40, 1.7863e-28, 1.4221e-33, 1.1277e-35, 1.5555e-38, 0.0000e+00,
         2.6588e-36, 2.1097e-35, 6.7883e-39, 1.8186e-35, 7.2920e-39, 2.8432e-38,
         9.5509e-40, 5.6698e-40, 8.1560e-41, 5.7726e-36, 0.0000e+00, 6.8155e-36,
         1.6441e-39, 5.1797e-38, 9.1402e-41, 5.4330e-38, 9.9305e-39, 3.3749e-41,
         2.3980e-30, 3.3743e-42, 2.3852e-32, 4.5391e-37, 3.1901e-38, 2.6555e-36,
         6.8935e-38, 2.1394e-39, 5.5685e-41, 9.6687e-39, 0.0000e+00, 8.7493e-40,
         9.4045e-36, 1.0365e-37, 2.7535e-35, 5.2848e-39, 1.5440e-40, 2.0249e-41,
         3.1646e-34, 2.9164e-40, 1.8110e-38, 2.8124e-42, 3.4065e-36, 7.1297e-38,
         1.6786e-38, 2.6446e-31, 0.0000e+00, 1.7918e-39, 3.1780e-41, 5.2890e-38,
         5.8616e-37, 5.6192e-43, 2.4104e-33, 7.7940e-15, 7.4730e-26, 1.5944e-30,
         3.1065e-29, 4.1376e-26, 7.3341e-27, 1.3160e-31, 5.7801e-30, 3.3449e-28,
         8.1288e-27, 5.3733e-24, 1.6446e-26, 6.7693e-30, 1.7419e-32, 2.2255e-29,
         5.1549e-32, 6.3842e-31, 2.8546e-31, 8.4764e-32, 3.3481e-25, 1.3744e-27,
         1.2825e-29, 1.3010e-32, 8.5522e-32, 4.1999e-27, 2.6561e-33, 2.5391e-28,
         2.5030e-29, 3.4420e-24, 1.1934e-25, 2.6204e-28, 1.6019e-29, 1.3563e-24,
         3.6413e-30, 1.2645e-28, 4.4894e-33, 4.3796e-30, 3.7951e-28, 3.3323e-22,
         4.1342e-27, 1.0300e-25, 3.2900e-26, 4.7741e-25, 4.5240e-26, 2.8945e-30,
         3.7797e-28, 4.1511e-26, 1.1907e-24, 9.1128e-15, 1.1499e-22, 1.9252e-25,
         1.5344e-24, 3.6028e-23, 1.7763e-19, 2.8124e-21, 5.8173e-19, 9.0711e-23,
         8.0103e-20, 3.1116e-23, 6.2617e-26, 1.3487e-25, 2.8245e-25, 2.6326e-23,
         2.0629e-18, 1.4949e-22, 3.3820e-23, 1.4597e-17, 5.5690e-20, 5.4107e-20,
         3.5353e-21, 8.1729e-26, 2.7802e-19, 1.7045e-25, 2.2840e-25, 6.2646e-20,
         1.3333e-17, 1.4997e-15, 1.1560e-24, 3.8457e-24, 6.6264e-22, 7.4117e-21,
         1.6747e-20, 9.7605e-21, 5.8352e-20, 7.0770e-17, 5.3944e-18, 1.5360e-16,
         7.4130e-21, 1.2811e-20, 9.1971e-21, 2.8468e-15, 1.2112e-15, 5.9355e-18,
         3.0055e-19, 4.6087e-15, 1.0000e+00, 1.7070e-28, 2.1192e-28, 1.5919e-28,
         1.9196e-28, 2.2293e-28, 2.3317e-28, 2.8672e-28, 3.1657e-28, 2.8153e-28,
         2.5195e-28, 3.6131e-28, 1.8182e-28, 2.9892e-28, 2.1955e-28, 1.5723e-28,
         1.7541e-28, 1.5954e-28, 4.0724e-28, 1.6594e-28, 1.7857e-28, 2.4620e-28,
         1.6114e-28, 2.2702e-28, 2.0316e-28, 2.3140e-28, 1.2141e-28, 1.6056e-28,
         1.2178e-28, 1.1598e-28, 2.1946e-28, 1.7480e-28, 2.2212e-28]])

In [136]:
torch.sum(lol)

tensor(1.)

In [135]:
logits = np.zeros((257), dtype = np.float32)  # Initialize all logits to 0
logits[224] = 1  # Make class 224 very confident

# Target index
target = 224

# Compute Cross-Entropy Loss
loss = cross_entropy(logits, target)
print(loss)

torch.Size([1, 257]) torch.Size([1])
tensor(4.5557)


In [149]:
def show_results_args(idx, cmd_idx):
    with h5py.File(h5_file, "r") as hf:
        pc_path = hf['pc_paths'][idx].decode("utf-8")
        args_logits = hf['args_logits'][idx]
        cmd_logits = hf['cmd_logits'][idx]
        seq_pred = hf['seq_pred'][idx]
        seq_target = hf['seq_target'][idx]
        z_pred = hf['z_pred'][idx]
        z_target = hf['z_target'][idx]

    #print(torch.tensor(args_logits[cmd_idx,0,:]))

    print(args_logits.shape, cmd_logits.shape, seq_pred.shape, seq_target.shape, z_pred.shape, z_target.shape)

    ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
    LINE_IDX = ALL_COMMANDS.index('Line')
    ARC_IDX = ALL_COMMANDS.index('Arc')
    CIRCLE_IDX = ALL_COMMANDS.index('Circle')
    EOS_IDX = ALL_COMMANDS.index('EOS')
    SOL_IDX = ALL_COMMANDS.index('SOL')
    EXT_IDX = ALL_COMMANDS.index('Ext')

    print(f"Point Cloud path: {pc_path}")

    predicted_args = []

    all_pred_commands = list(seq_pred[:, 0])                # All predicted commands
    seq_length = list(seq_target[:, 0]).index(EOS_IDX) + 3  # Sequence length of the sample
    
    all_pred_args = list(seq_pred[:seq_length, 1:])         # All predicted args for predicted commands, shape = (seq_length, 16)
    all_trgt_args = list(seq_target[:seq_length, 1:])       # All target    args for target    commands, shape = (seq_length, 16)
    print("predicted")
    for a in all_trgt_args:
        print(a)
    
    pred_cmd_args = all_pred_args[cmd_idx]                               # Predicted args for a command, shape = (16)
    print("pred first arg: ", pred_cmd_args[0])
    print("pred args +1  : ", pred_cmd_args + 1)
    target_args = all_trgt_args[cmd_idx]                                 # Target    args for a command, shape = (16)
    print("trgt first arg: ", pred_cmd_args[0])
    cmd_arg_logits = args_logits[cmd_idx, :, :]                          # Logits of all 16 args for a command, shape = (16, 257)
    print(cmd_arg_logits.shape)
    cmd_arg_softmax = softmax(cmd_arg_logits)           # Softmax of all 16 args for a command, shape = (16, 257)

   # for j, b in enumerate(cmd_arg_softmax[0]):
    #    print(j, round(b, 8))

    pred_sm = list(cmd_arg_softmax[torch.arange(cfg.n_args), pred_cmd_args + 1])   # Softmax of predicted args
    trgt_sm = list(cmd_arg_softmax[torch.arange(cfg.n_args), target_args + 1])     # Softmax of target args

    arg_loss = []
    for i, logits in enumerate(cmd_arg_logits):
        if i == 0:
            print(np.argmax(logits))
            print(target_args[i] + 1)
        arg_loss.append(cross_entropy(logits, target_args[i] + 1).item())

    #print(len(target_args), len(trgt_sm), len(pred_cmd_args), len(pred_sm), len(arg_loss))
    df = pd.DataFrame(list(zip(target_args, pred_cmd_args, trgt_sm, pred_sm, arg_loss)),
                      columns=['trgt', 'pred','prob_trgt', 'prob_pred', 'loss'])
    return df
        


show_results_args(0,1) 

(60, 16, 257) (60, 6) (60, 17) (60, 17) (256,) (256,)
Point Cloud path: experiments/best_5_results/infered_point_clouds/00250456.ply
predicted
[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[223 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[223 223  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[128 223  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[128 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[ -1  -1  -1  -1  -1 192  64 192  32 128  32 192 224 128   0   1]
[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
pred first arg:  223
pred args +1  :  [224 129   0   0   0   0   0   0   0   0   0   0   0   0   0   0]
trgt first arg:  223
(16, 257)
224
224
torch.Size([1, 257]) torch.Size([1])
torch.Size([1, 257]) torch.Size([1])
torch.Size([1, 257]) torch.Size([1])
torch.Size([1, 257]) torch.Size([1])
torch.Size([1, 257]) torch.Size([1])

,trgt,pred,prob_trgt,prob_pred,loss
0,223,223,1.000000e+00,1.000000e+00,0.000000
1,128,128,1.000000e+00,1.000000e+00,0.000000
2,-1,-1,8.102734e-13,8.102734e-13,27.841404
3,-1,-1,2.104556e-09,2.104556e-09,19.979164
4,-1,-1,5.883002e-22,5.883002e-22,48.884804
5,-1,-1,1.155967e-13,1.155967e-13,29.788673
6,-1,-1,7.749392e-13,7.749392e-13,27.885992
7,-1,-1,4.823587e-13,4.823587e-13,28.360088
8,-1,-1,9.554483e-18,9.554483e-18,39.189518
9,-1,-1,1.278187e-17,1.278187e-17,38.898506


In [145]:
abc = torch.tensor([-1.1593e+01, -1.1793e+01, -1.1749e+01, -1.1726e+01, -1.2273e+01,
        -1.1608e+01, -1.1866e+01, -1.1998e+01, -1.1636e+01, -1.2013e+01,
        -1.1954e+01, -1.1791e+01, -1.1933e+01, -1.1549e+01, -1.1704e+01,
        -1.2033e+01, -1.1244e+01, -1.1987e+01, -1.1347e+01, -1.1576e+01,
        -1.1921e+01, -1.1921e+01, -1.1136e+01, -1.1712e+01, -1.2110e+01,
        -1.1784e+01, -1.2137e+01, -1.1776e+01, -1.1691e+01, -1.1797e+01,
        -1.2182e+01, -1.1932e+01, -1.1905e+01, -1.1734e+01, -1.1973e+01,
        -1.2023e+01, -1.2118e+01, -2.3991e+01, -1.8785e+01, -1.1634e+01,
        -1.2095e+01, -1.1776e+01, -1.2037e+01, -1.2041e+01, -1.1660e+01,
        -2.7247e+01, -3.2370e+01, -3.6731e+01, -1.4870e+01, -1.1775e+01,
        -3.2772e+01, -2.5371e+01, -1.8860e+01, -3.5922e+01, -1.1875e+01,
        -1.8692e+01, -4.6048e+01, -3.2303e+01, -1.1452e+01, -1.1416e+01,
        -1.1846e+01, -2.4532e+01, -2.0030e+01, -3.1591e+01, -1.1655e+01,
        -3.6347e+01, -4.7404e+01, -4.2596e+01, -3.6891e+01, -1.9054e+01,
        -1.1956e+01, -2.7473e+01, -4.0038e+01, -1.1881e+01, -2.3622e+01,
        -2.8459e+01, -3.5045e+01, -4.9315e+01, -2.9904e+01, -2.7833e+01,
        -3.5874e+01, -2.7981e+01, -3.5803e+01, -3.4442e+01, -3.7835e+01,
        -3.8357e+01, -4.0295e+01, -2.9129e+01, -4.9995e+01, -2.8963e+01,
        -3.7292e+01, -3.3842e+01, -4.0183e+01, -3.3794e+01, -3.5494e+01,
        -4.1179e+01, -1.6192e+01, -4.3450e+01, -2.0802e+01, -3.1672e+01,
        -3.4327e+01, -2.9905e+01, -3.3556e+01, -3.7029e+01, -4.0676e+01,
        -3.5521e+01, -4.6741e+01, -3.7923e+01, -2.8641e+01, -3.3149e+01,
        -2.7566e+01, -3.6125e+01, -3.9658e+01, -4.1691e+01, -2.5125e+01,
        -3.9021e+01, -3.4893e+01, -4.3622e+01, -2.9656e+01, -3.3523e+01,
        -3.4969e+01, -1.8396e+01, -4.8976e+01, -3.7206e+01, -4.1237e+01,
        -3.3821e+01, -3.1416e+01, -4.5186e+01, -2.3094e+01,  1.9526e+01,
        -5.8446e+00, -1.6600e+01, -1.3630e+01, -6.4358e+00, -8.1660e+00,
        -1.9094e+01, -1.5312e+01, -1.1254e+01, -8.0631e+00, -1.5693e+00,
        -7.3584e+00, -1.5154e+01, -2.1116e+01, -1.3964e+01, -2.0031e+01,
        -1.7515e+01, -1.8320e+01, -1.9534e+01, -4.3450e+00, -9.8405e+00,
        -1.4515e+01, -2.1408e+01, -1.9525e+01, -8.7235e+00, -2.2997e+01,
        -1.1529e+01, -1.3846e+01, -2.0147e+00, -5.3765e+00, -1.1498e+01,
        -1.4293e+01, -2.9460e+00, -1.5774e+01, -1.2226e+01, -2.2472e+01,
        -1.5589e+01, -1.1127e+01,  2.5581e+00, -8.7392e+00, -5.5238e+00,
        -6.6651e+00, -3.9901e+00, -6.3465e+00, -1.6003e+01, -1.1131e+01,
        -6.4326e+00, -3.0762e+00,  1.9682e+01,  1.4941e+00, -4.8983e+00,
        -2.8226e+00,  3.3351e-01,  8.8367e+00,  4.6910e+00,  1.0023e+01,
         1.2569e+00,  8.0403e+00,  1.8695e-01, -6.0215e+00, -5.2542e+00,
        -4.5150e+00,  1.9806e-02,  1.1289e+01,  1.7565e+00,  2.7028e-01,
         1.3246e+01,  7.6768e+00,  7.6479e+00,  4.9198e+00, -5.7551e+00,
         9.2847e+00, -5.0201e+00, -4.7274e+00,  7.7945e+00,  1.3155e+01,
         1.7878e+01, -3.1058e+00, -1.9038e+00,  3.2455e+00,  5.6601e+00,
         6.4752e+00,  5.9353e+00,  7.7235e+00,  1.4824e+01,  1.2250e+01,
         1.5599e+01,  5.6602e+00,  6.2073e+00,  5.8759e+00,  1.8519e+01,
         1.7664e+01,  1.2346e+01,  9.3626e+00,  1.9000e+01,  5.2011e+01,
        -1.1926e+01, -1.1710e+01, -1.1996e+01, -1.1809e+01, -1.1659e+01,
        -1.1614e+01, -1.1408e+01, -1.1309e+01, -1.1426e+01, -1.1537e+01,
        -1.1177e+01, -1.1863e+01, -1.1366e+01, -1.1675e+01, -1.2009e+01,
        -1.1899e+01, -1.1994e+01, -1.1057e+01, -1.1955e+01, -1.1881e+01,
        -1.1560e+01, -1.1984e+01, -1.1641e+01, -1.1752e+01, -1.1622e+01,
        -1.2267e+01, -1.1988e+01, -1.2264e+01, -1.2313e+01, -1.1675e+01,
        -1.1903e+01, -1.1663e+01])

In [147]:
torch.argmax(abc)
torch.sum(abc)

tensor(-3778.5574)

In [26]:
show_results_args(0, 1)

(60, 16, 257) (60, 6) (60, 17) (60, 17) (256,) (256,)
Point Cloud path: experiments/best_5_results/infered_point_clouds/00575185.ply
(60, 16, 257)
[204 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[206 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[[  4.936258   -11.334982   -12.412564   -21.583628    -4.397084
  -14.8361     -16.388008   -16.743713    -0.03594986 -12.276738
   -9.990585    -6.9201374   -7.9274597    1.7885818  -17.452883
  -17.203762  ]
 [-34.934937    17.211254     3.4887717  -21.455511    -4.319825
  -14.038717   -16.694796   -16.840965     9.060187     5.083467
   10.411677    -8.987089    -5.594931    -8.011899   -17.759392
  -16.623045  ]
 [-16.667828   -13.389228   -20.159206   -21.528215    -4.7289324
   -0.30675676 -15.744706    -3.0173824   -3.5703645  -20.636581
  -20.985083    -8.817478   -10.37203     -3.1957917  -18.198559
  -17.071405  ]
 [-16.667828   -13.389228   -20.159206   -21.528215    -4.7289324
   -0.30675676 -15.74

In [ ]:
export2step()

In [22]:
step2stl()

### Gedanken

Was noch wichtig/Meeting morgen:

- args loss visualization

Meeting: 
- had to finish applications
- first thing I did was refactor training
    - Automatic resume of training if cluster fails
    - Parallelization (30mins/epoch -> 12 mins/epoch)
- implemented test script
- Implemented cosine annealing learning rate -> show new training with better convergence!
- Worked on CAD Loss understanding
- Implemented testing pipeline to make sure the data is alligned
- Finished pc2cad pipeline with thorough understanding of loss for train/val/test
- Created interactive pc2cad

HiWi:
- created SAiL poster
- documented literature research
- made Blensor work

Next:

- train DeepCAD ourselves?
- Train both models in one pipeline using CADLoss?
- use blensor to create new data?
